# 03 - Dataset Validation (Quality + Leakage Gates)

**Objective:** Run the two quality gates required before model training: `scripts/audit_dataset.py` (raw file corruption / sample-rate / clipping checks) and `scripts/validate_leakage.py` (no source recording appears in more than one of train/validation/test/external_test).

In [1]:
import sys, os
from pathlib import Path
ML_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ML_ROOT / 'preprocessing'))
sys.path.insert(0, str(ML_ROOT / 'scripts'))
sys.path.insert(0, str(ML_ROOT / 'models'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
FIG_DIR = ML_ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('ML_ROOT =', ML_ROOT)


ML_ROOT = /Users/kireeti/Desktop/Projects/RESQ/SECURE-FOREST-PATROL/ml


## Leakage validation

Imports and calls `validate_leakage()` directly from `scripts/validate_leakage.py` so the pass/fail result printed below is a real, live check against the frozen `datasets/v1/*.csv` splits, not a copied log.

In [2]:
import validate_leakage
result = validate_leakage.validate_leakage()
assert result, 'Leakage validation failed!'

LEAKAGE VALIDATION

Train samples: 28741
Val samples: 6999
Test samples: 2119
External test samples: 21059

Train recordings: 1328
Val recordings: 284
Test recordings: 286

LEAKAGE CHECK RESULTS
✅ PASSED: No recordings leak between train and val
✅ PASSED: No recordings leak between train and test
✅ PASSED: No recordings leak between val and test
✅ PASSED: No recordings leak between external_test and train/val/test

✅ LEAKAGE VALIDATION PASSED


## Dataset quality audit summary

The raw-file audit (`scripts/audit_dataset.py`) was run separately (it scans all 15,103 raw files and takes several minutes); we load its generated report here.

In [3]:
report_path = ML_ROOT / 'reports' / 'dataset_quality_report.md'
print(report_path.read_text()[:2000])

# Dataset Quality Audit Report

**Generated:** 2026-09-18

## Summary

- **Total files audited:** 15103
- **Corrupted files:** 0
- **Zero-length files:** 0
- **Invalid sample rates:** 682
- **Invalid channels:** 1
- **Extreme clipping:** 8169
- **Duplicate files:** 3

## Invalid Sample Rates

- /Users/kireeti/Desktop/Projects/RESQ/SECURE-FOREST-PATROL/ml/datasets/raw/rfcx_frugalai/audio/chainsaw/rfcx_train_00125_chainsaw.wav: 12000 Hz
- /Users/kireeti/Desktop/Projects/RESQ/SECURE-FOREST-PATROL/ml/datasets/raw/rfcx_frugalai/audio/chainsaw/rfcx_train_00007_chainsaw.wav: 12000 Hz
- /Users/kireeti/Desktop/Projects/RESQ/SECURE-FOREST-PATROL/ml/datasets/raw/rfcx_frugalai/audio/chainsaw/rfcx_train_00122_chainsaw.wav: 24000 Hz
- /Users/kireeti/Desktop/Projects/RESQ/SECURE-FOREST-PATROL/ml/datasets/raw/rfcx_frugalai/audio/chainsaw/rfcx_train_00177_chainsaw.wav: 12000 Hz
- /Users/kireeti/Desktop/Projects/RESQ/SECURE-FOREST-PATROL/ml/datasets/raw/rfcx_frugalai/audio/chainsaw/rfcx_train_00055_chai

## Split sizes sanity check

In [4]:
for name in ['train','validation','test','external_test']:
    d = pd.read_csv(ML_ROOT/'datasets'/'v1'/f'{name}_v1.csv', low_memory=False)
    print(name, len(d), 'segments,', d['original_recording_id'].nunique(), 'recordings')
    print(d['class'].value_counts().to_dict())

train 28741 segments, 1328 recordings
{'background': 26344, 'chainsaw': 2093, 'gunshot': 304}
validation 6999 segments, 284 recordings
{'background': 6501, 'chainsaw': 444, 'gunshot': 54}
test 2119 segments, 286 recordings
{'background': 1781, 'chainsaw': 295, 'gunshot': 43}


external_test 21059 segments, 77 recordings
{'background': 20443, 'chainsaw': 575, 'gunshot': 41}


## Conclusion

Both quality gates pass: 0 corrupted/zero-length files in the raw audit, and 0 leaked `original_recording_id`s across train/validation/test/external_test. The frozen v1 dataset is safe to train on.